# 03. Fitting, Interpreting, and Testing a Multiple Regression Model

A fitted multiple regression table gives several different pieces of evidence. This notebook separates them into coefficient-level inference and model-level inference.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


In [ ]:
delivery = pd.read_csv("data/delivery_time.csv")
model = smf.ols("Time ~ Cases + Distance", data=delivery).fit()
print(model.summary())


## Coefficient Interpretation

For this fitted model:

- the `Cases` coefficient is the expected change in delivery time for one additional stocked case, holding walking distance fixed;
- the `Distance` coefficient is the expected change in delivery time for one additional unit of walking distance, holding the number of cases fixed.

The phrase “holding the other predictors fixed” is part of the interpretation, not a footnote.


In [ ]:
coef_table = pd.DataFrame({
    "estimate": model.params,
    "std_error": model.bse,
    "t_stat": model.tvalues,
    "p_value": model.pvalues,
})
coef_table


## Individual t Tests

For predictor $j$, the usual test is

$$H_0:\beta_j=0 \quad \text{versus} \quad H_a:\beta_j\ne 0,$$

using

$$t=\frac{b_j}{s\{b_j\}}$$

with $n-(k+1)$ degrees of freedom.


In [ ]:
alpha = 0.05
ci = model.conf_int(alpha=alpha)
ci.columns = ["lower", "upper"]
ci


## Overall F Test

The overall regression test asks whether the model with predictors improves over an intercept-only model:

$$H_0:\beta_1=\beta_2=\cdots=\beta_k=0.$$

This is different from asking whether every individual coefficient is significant. The F test is a joint test.


In [ ]:
from checks import model_snapshot
model_snapshot(model)


## R-squared and Adjusted R-squared

$R^2$ is the fraction of total response variation explained by the fitted model. It cannot decrease when you add predictors, so it is not a sufficient model-selection rule. Adjusted $R^2$ includes a penalty for adding predictors and is usually more useful for comparing models with different numbers of predictors.


In [ ]:
print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")
